# AdaBoost：加权投票的 Boosting 先驱
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：07_集成学习 | 源文件：AdaBoost.py | 核心功能：样本权重调整、弱分类器加权组合

## 概述

AdaBoost（Adaptive Boosting）是最早的 Boosting 算法。每轮训练后，增加被错误分类样本的权重，让下一个模型更关注"难"样本。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
# --- 导入库 ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

## 数学原理

### 1. AdaBoost 算法概述

AdaBoost（Adaptive Boosting）通过**调整样本权重**来关注难分类样本。每一轮训练后，增加被错误分类样本的权重，减少正确分类样本的权重。

### 2. 样本权重初始化

设训练集 $\{(x_i, y_i)\}_{i=1}^n$，$y_i \in \{-1, +1\}$。初始权重：

$$w_i^{(1)} = \frac{1}{n}, \quad i=1,\ldots,n$$

### 3. 第 $m$ 轮的迭代过程

**步骤一**：用加权训练集训练弱学习器 $f_m(x)$

**步骤二**：计算加权错误率：

$$\epsilon_m = \frac{\sum_{i=1}^{n} w_i^{(m)} \mathbb{I}[y_i \neq f_m(x_i)]}{\sum_{i=1}^{n} w_i^{(m)}}$$

**步骤三**：计算学习器权重（$\alpha_m$）：

$$\alpha_m = \frac{1}{2}\ln\left(\frac{1-\epsilon_m}{\epsilon_m}\right)$$

- 当 $\epsilon_m < 0.5$（比随机猜测好）：$\alpha_m > 0$
- 当 $\epsilon_m = 0$（完美分类）：$\alpha_m \to \infty$
- 当 $\epsilon_m \geq 0.5$（不比猜测好）：终止

**步骤四**：更新样本权重：

$$w_i^{(m+1)} = w_i^{(m)} \cdot \exp\left(-\alpha_m y_i f_m(x_i)\right)$$

归一化使 $\sum w_i = 1$。被错误分类的样本（$y_i f_m(x_i) < 0$）权重增大。

### 4. 最终分类器

$$F(x) = \text{sign}\left(\sum_{m=1}^{M} \alpha_m f_m(x)\right)$$

这是 $M$ 个弱学习器的**加权投票**，权重 $\alpha_m$ 由每个学习器的准确度决定。

### 5. 指数损失函数

AdaBoost 等价于在加法模型上最小化**指数损失**：

$$L(y, F(x)) = \exp(-y F(x))$$

对 $F$ 求导：

$$\frac{\partial L}{\partial F} = -y \exp(-y F(x))$$

每一步通过前向分步加法模型拟合负梯度，推导出 AdaBoost 的权重更新公式。

### 6. 学习率的作用

代码中 `learning_rate` 用于正则化 $\alpha_m$：

$$F(x) = \sum_{m=1}^{M} \nu \cdot \alpha_m f_m(x), \quad 0 < \nu \leq 1$$

较小的 $\nu$ 使每个弱学习器的贡献缩小，需要更多轮次但泛化更好。

### 7. 基学习器复杂度

代码对比了不同树深度（1, 2, 3, 5）的效果：
- 深度=1（决策树桩）：最简单的弱学习器，AdaBoost 的经典选择
- 深度增大 → 单个学习器更强 → $\epsilon_m$ 更小 → $\alpha_m$ 更大
- 但过强的基学习器可能导致过拟合

### 8. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `AdaBoostClassifier(n_estimators=50)` | $M=50$ 轮迭代 |
| `learning_rate=1.0` | $\nu=1$，不对 $\alpha_m$ 缩放 |
| `DecisionTreeClassifier(max_depth=1)` | 决策树桩作为弱学习器 $f_m$ |
| 样本权重（隐式） | $w_i^{(m)}$，由 `sklearn` 内部管理 |
| `feature_importances_` | $\sum_m \alpha_m \cdot (\text{特征在 } f_m \text{ 中的重要性})$ |

### 1. 生成数据

生成合成数据集，用于演示算法的核心行为。

In [ ]:
X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                           n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 2. 基准：决策树桩

运行 2. 基准：决策树桩 的代码，观察算法在该环节的行为。

In [ ]:
dt_stump = DecisionTreeClassifier(max_depth=1, random_state=42)
dt_stump.fit(X_train, y_train)
print(f"=== 决策树桩: 测试准确率={dt_stump.score(X_test, y_test):.4f} ===")

### 3. AdaBoost 分类

在分类任务上训练模型并评估性能。

In [ ]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=50,
    learning_rate=1.0,
    random_state=42,
# --- 继续 ---
)
ada.fit(X_train, y_train)

print(f"\n=== AdaBoost ===")
print(f"训练准确率: {ada.score(X_train, y_train):.4f}")
print(f"测试准确率: {ada.score(X_test, y_test):.4f}")

### 4. n_estimators 对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== n_estimators 对比 ===")
for n in [5, 10, 25, 50, 100, 200]:
    a = AdaBoostClassifier(n_estimators=n, random_state=42)
    a.fit(X_train, y_train)
    print(f"  n={n:>3}: 测试准确率={a.score(X_test, y_test):.4f}")

### 5. learning_rate 对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== learning_rate 对比 ===")
for lr in [0.01, 0.1, 0.5, 1.0, 2.0]:
    a = AdaBoostClassifier(n_estimators=50, learning_rate=lr, random_state=42)
    a.fit(X_train, y_train)
    print(f"  lr={lr:>4}: 测试准确率={a.score(X_test, y_test):.4f}")

### 6. 基学习器复杂度

运行 6. 基学习器复杂度 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 基学习器复杂度影响 ===")
for depth in [1, 2, 3, 5]:
    a = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=depth),
        n_estimators=50, random_state=42,
# --- 继续 ---
    )
    a.fit(X_train, y_train)
    print(f"  树深度={depth}: 测试准确率={a.score(X_test, y_test):.4f}")

### 7. 特征重要性

分析各特征对模型预测的贡献度。

In [ ]:
print("\n=== 特征重要性 ===")
importances = ada.feature_importances_
for i in np.argsort(importances)[::-1]:
    print(f"  特征{i}: {importances[i]:.4f}")

### 8. AdaBoost 原理

运行 8. AdaBoost 原理 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== AdaBoost 算法原理 ===")
print("1. 初始化样本权重: w_i = 1/n")
print("2. 对每轮 t = 1, ..., T:")
print("   a. 用加权数据训练弱学习器 h_t")
print("   b. 计算加权错误率: ε_t = Σ w_i × I(h_t(x_i) ≠ y_i)")
# --- 输出结果 ---
print("   c. 计算学习器权重: α_t = 0.5 × ln((1-ε_t)/ε_t)")
print("   d. 更新样本权重: w_i ×= exp(±α_t)（错误样本权重增加）")
print("3. 最终: H(x) = sign(Σ α_t × h_t(x))")

### 9. AdaBoost 回归

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print("\n=== AdaBoost 回归 ===")
from sklearn.datasets import make_regression
X_r, y_r = make_regression(n_samples=300, n_features=10, noise=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)
ada_r = AdaBoostRegressor(n_estimators=50, random_state=42)
# --- 训练模型 ---
ada_r.fit(X_tr, y_tr)
print(f"R²: {ada_r.score(X_te, y_te):.4f}")

print("\n=== AdaBoost 要点 ===")
print("- 经典 Boosting：通过样本权重调整关注难分类样本")
print("- 基学习器通常是弱学习器（决策树桩）")
print("- learning_rate: 缩小每轮的贡献，提高泛化")
print("- 对异常值敏感（异常值可能被赋予极大权重）")
# --- 输出结果 ---
print("- 与 GradientBoosting 的区别: AdaBoost 用权重调整，GB 用残差拟合")

## 关键代码解释

```python
from sklearn.ensemble import AdaBoostClassifier
ada = AdaBoostClassifier(n_estimators=50, learning_rate=1.0, algorithm="SAMME")
```

每个弱分类器有一个权重 alpha = 0.5 * ln((1-err)/err)，准确率越高权重越大。

## 注意事项

1. 对噪声和异常值敏感（权重会被难样本主导）
2. 基分类器不能太强（通常用决策树桩 depth=1）
3. SAMME vs SAMME.R：SAMME.R 用概率输出，通常更好

## 总结与延伸

以上代码展示了 **AdaBoost** 的完整流程。

**解读要点**：
- 对比 **单模型 vs 集成模型** 的性能提升
- 关注 **特征重要性** 排名
- 观察 OOB 分数（随机森林）或 early stopping 轮次（XGBoost）

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **AdaBoost.R2**：回归版本
- **Gradient Boosting**：更通用的 Boosting 框架
- **AdaBoost 与 SVM 的联系**：两者都关注"难"样本
